In [1]:

import qiskit_metal.toolbox_metal.dsl as dsl
import qiskit_metal.toolbox_metal.dsl.builder as dslb


12:10AM 24s WARNING [_maybe_warn_import_rename]: [FutureWarning] A future major release of quantum-metal (target v0.8 / v1.0) will rename the Python import path from `qiskit_metal` to `quantum_metal` to match the PyPI package name. Plan to update your imports ahead of that release. See ROADMAP.md and the README rebrand notice. Suppress this warning with QISKIT_METAL_SUPPRESS_RENAME_WARNING=1.


In [2]:
spec, base_dir = dslb._load_yaml("../chain_2q_native.metal.yaml")

spec

{'schema': 'qiskit-metal/design-dsl/3',
 'vars': {'qx': '1.2mm',
  'pad_w': '420um',
  'pad_h': '90um',
  'trace_w': '12um',
  'trace_gap': '7um',
  'bus_attach': '0.34mm',
  'c_q': '65fF',
  'ej_q': '18GHz'},
 'hamiltonian': {'subsystems': {'Q1': {'model': 'transmon',
    'EJ': '${ej_q}',
    'C': '${circuit.Q1.C}'},
   'Q2': {'model': 'transmon', 'EJ': '${ej_q}', 'C': '${circuit.Q2.C}'}},
  'couplings': [{'from': 'Q1', 'to': 'Q2', 'g': '12MHz'}]},
 'circuit': {'Q1': {'type': 'transmon',
   'C': '${c_q}',
   'pad_width': '${pad_w}'},
  'Q2': {'type': 'transmon', 'C': '${c_q}', 'pad_width': '${pad_w}'},
  'bus': {'width': '${trace_w}', 'gap': '${trace_gap}'}},
 'netlist': {'connections': [{'from': 'Q1.bus', 'to': 'bus.start'},
   {'from': 'bus.end', 'to': 'Q2.bus'}]},
 'geometry': {'design': {'class': 'DesignPlanar',
   'chip': {'size': '6mm x 6mm'}},
  'templates': {'transmon_pad_pair': {'primitives': [{'name': 'pad_left',
      'type': 'poly.rectangle',
      'center': ['-0.12mm', '0

In [3]:
spec = dslb._expand_includes(spec, base_dir)



geometry_spec = spec.get("geometry")

design_spec = geometry_spec.get("design")

geometry_spec


{'design': {'class': 'DesignPlanar', 'chip': {'size': '6mm x 6mm'}},
 'templates': {'transmon_pad_pair': {'primitives': [{'name': 'pad_left',
     'type': 'poly.rectangle',
     'center': ['-0.12mm', '0mm'],
     'size': ['${circuit.Q1.pad_width}', '${pad_h}'],
     'layer': 1},
    {'name': 'pad_right',
     'type': 'poly.rectangle',
     'center': ['0.12mm', '0mm'],
     'size': ['${circuit.Q1.pad_width}', '${pad_h}'],
     'layer': 1},
    {'name': 'pocket',
     'type': 'poly.rectangle',
     'center': ['0mm', '0mm'],
     'size': ['800um', '520um'],
     'subtract': True,
     'layer': 1},
    {'name': 'jj',
     'type': 'junction.line',
     'points': [['0mm', '-45um'], ['0mm', '45um']],
     'width': '10um',
     'layer': 1}]}},
 'components': {'Q1': {'$extend': 'transmon_pad_pair',
   'translate': ['-${qx}', '0mm'],
   'pins': [{'name': 'bus',
     'points': [['0.34mm', '-6um'], ['0.34mm', '6um']],
     'width': '${circuit.bus.width}',
     'gap': '${circuit.bus.gap}'}]},
  'Q2

In [4]:
design_spec

{'class': 'DesignPlanar', 'chip': {'size': '6mm x 6mm'}}

In [7]:
vars_table = dslb._optional_mapping(spec, "vars")
vars_table


{'qx': '1.2mm',
 'pad_w': '420um',
 'pad_h': '90um',
 'trace_w': '12um',
 'trace_gap': '7um',
 'bus_attach': '0.34mm',
 'c_q': '65fF',
 'ej_q': '18GHz'}

In [8]:
ctx_vars = {**vars_table, "vars": vars_table}
ctx_vars

{'qx': '1.2mm',
 'pad_w': '420um',
 'pad_h': '90um',
 'trace_w': '12um',
 'trace_gap': '7um',
 'bus_attach': '0.34mm',
 'c_q': '65fF',
 'ej_q': '18GHz',
 'vars': {'qx': '1.2mm',
  'pad_w': '420um',
  'pad_h': '90um',
  'trace_w': '12um',
  'trace_gap': '7um',
  'bus_attach': '0.34mm',
  'c_q': '65fF',
  'ej_q': '18GHz'}}

In [9]:
circuit = dslb._walk_substitute(dslb._optional_mapping(spec, "circuit"), ctx_vars)
circuit

{'Q1': {'type': 'transmon', 'C': '65fF', 'pad_width': '420um'},
 'Q2': {'type': 'transmon', 'C': '65fF', 'pad_width': '420um'},
 'bus': {'width': '12um', 'gap': '7um'}}

In [10]:
netlist_spec = spec["netlist"]
netlist_spec

{'connections': [{'from': 'Q1.bus', 'to': 'bus.start'},
  {'from': 'bus.end', 'to': 'Q2.bus'}]}

In [12]:
hamiltonian = dslb._walk_substitute(
        dslb._optional_mapping(spec , "hamiltonian"),
        {**ctx_vars, "circuit": circuit},
    )
hamiltonian

{'subsystems': {'Q1': {'model': 'transmon', 'EJ': '18GHz', 'C': '65fF'},
  'Q2': {'model': 'transmon', 'EJ': '18GHz', 'C': '65fF'}},
 'couplings': [{'from': 'Q1', 'to': 'Q2', 'g': '12MHz'}]}

In [13]:
netlist = dslb._walk_substitute(
        dict(netlist_spec or {}),
        {**ctx_vars, "circuit": circuit, "hamiltonian": hamiltonian},
    )
netlist

{'connections': [{'from': 'Q1.bus', 'to': 'bus.start'},
  {'from': 'bus.end', 'to': 'Q2.bus'}]}

In [15]:
ctx = {
        **vars_table,
        "vars": vars_table,
        "circuit": circuit,
        "hamiltonian": hamiltonian,
        "netlist": netlist,
    }
ctx

{'qx': '1.2mm',
 'pad_w': '420um',
 'pad_h': '90um',
 'trace_w': '12um',
 'trace_gap': '7um',
 'bus_attach': '0.34mm',
 'c_q': '65fF',
 'ej_q': '18GHz',
 'vars': {'qx': '1.2mm',
  'pad_w': '420um',
  'pad_h': '90um',
  'trace_w': '12um',
  'trace_gap': '7um',
  'bus_attach': '0.34mm',
  'c_q': '65fF',
  'ej_q': '18GHz'},
 'circuit': {'Q1': {'type': 'transmon', 'C': '65fF', 'pad_width': '420um'},
  'Q2': {'type': 'transmon', 'C': '65fF', 'pad_width': '420um'},
  'bus': {'width': '12um', 'gap': '7um'}},
 'hamiltonian': {'subsystems': {'Q1': {'model': 'transmon',
    'EJ': '18GHz',
    'C': '65fF'},
   'Q2': {'model': 'transmon', 'EJ': '18GHz', 'C': '65fF'}},
  'couplings': [{'from': 'Q1', 'to': 'Q2', 'g': '12MHz'}]},
 'netlist': {'connections': [{'from': 'Q1.bus', 'to': 'bus.start'},
   {'from': 'bus.end', 'to': 'Q2.bus'}]}}

In [17]:
resolved_geometry = dict(geometry_spec)
resolved_geometry["design"] = dslb._walk_substitute(geometry_spec["design"], ctx)

resolved_geometry


{'design': {'class': 'DesignPlanar', 'chip': {'size': '6mm x 6mm'}},
 'templates': {'transmon_pad_pair': {'primitives': [{'name': 'pad_left',
     'type': 'poly.rectangle',
     'center': ['-0.12mm', '0mm'],
     'size': ['${circuit.Q1.pad_width}', '${pad_h}'],
     'layer': 1},
    {'name': 'pad_right',
     'type': 'poly.rectangle',
     'center': ['0.12mm', '0mm'],
     'size': ['${circuit.Q1.pad_width}', '${pad_h}'],
     'layer': 1},
    {'name': 'pocket',
     'type': 'poly.rectangle',
     'center': ['0mm', '0mm'],
     'size': ['800um', '520um'],
     'subtract': True,
     'layer': 1},
    {'name': 'jj',
     'type': 'junction.line',
     'points': [['0mm', '-45um'], ['0mm', '45um']],
     'width': '10um',
     'layer': 1}]}},
 'components': {'Q1': {'$extend': 'transmon_pad_pair',
   'translate': ['-${qx}', '0mm'],
   'pins': [{'name': 'bus',
     'points': [['0.34mm', '-6um'], ['0.34mm', '6um']],
     'width': '${circuit.bus.width}',
     'gap': '${circuit.bus.gap}'}]},
  'Q2